# Домашнее задание: В поисках лучшей сети

**NB! [Ссылка на частично решенный семинар](https://colab.research.google.com/drive/1XWP1BQFcOGwViswa6_YfVxYIzGiH4MiS?usp=sharing)**

В этом задании вы создадите сеть для решения задачи классификации изображений CIFAR10.

Эта тетрадка является продолжением семинара. Пожалуйста, попробуйте выполнить его, если вы еще этого не сделали.

(пожалуйста, прочтите его хотя бы по диагонали)

* Главная задача — создать сеть с максимально высокой __точностью__ (accuracy), какой вы только сможете добиться.
* В конце есть __мини-отчет__, который вам нужно будет заполнить. Я рекомендую сначала заполнять его по мере итераций.

## Оценивание
* начальный балл — 0
* +4 б за описание вашего итерационного пути в отчете ниже.
* +4 б за создание сети, которая достигает точности выше 20%
* за преодоление каждого из этих порогов на __ТЕСТОВОМ__ наборе данных:
    * 50% (10 total)
    * 60% (12 total)
    * 65% (14 total)
    * 70% (16 total)
    * 75% (18 total)
    * 80% (20 total)
    
## Ограничения
* Пожалуйста, НЕ используйте предобученные сети для этого задания, пока не достигнете 80%.
 * Другими словами, базовые пороги должны быть преодолены без предобученных сетей. После этого вы можете использовать все, что захотите.
* вы __можете__ использовать валидационные данные для обучения, но вы __не можете__ делать ничего с тестовыми данными, кроме как запускать процедуру оценки.

## Советы о том, что можно сделать:

 * __Размер сети__
   * БОЛЬШЕ нейронов,
   * БОЛЬШЕ слоев, ([документация torch.nn](http://pytorch.org/docs/master/nn.html))

   * Нелинейности в скрытых слоях
     * tanh, relu, leaky relu и т.д.
   * Большие сети могут требовать больше эпох для обучения, так что не выкидывайте свою сетку только потому, что она не превзошла базовое решение за 5 эпох.

   * Увеличение каналов 32 -> 64 -> 128 -> 256 -> 512 и т.п. Почитайте про VGG сеть.

   * Пх’нглуи мглв’нафх Ктулху Р’льех вгах’нагл фхтагн!


### Главное правило прототипирования: одно изменение за раз
   * К этому моменту у вас, вероятно, есть несколько идей, что можно сделать. Обязательно попробуйте их! Но есть одна загвоздка: __никогда не тестируйте несколько новых вещей одновременно__.


### Оптимизация
   * Обучение в течение 100 эпох, несмотря на то, что лосс не двигается — вероятно, плохая идея.
   * Некоторые сети сходятся за 5 эпох, другие — за 500. 
   * Правильный путь: остановиться, когда показатель на валидации не улучшается в течение 10 итераций после достижения максимума.
   * Вам определенно стоит использовать адаптивные оптимизаторы
     * rmsprop, nesterov_momentum, adam, adagrad и так далее.
     * Они сходятся быстрее и иногда достигают лучших оптимумов
     * Может иметь смысл настроить скорость обучения/моментум, другие параметры обучения, размер батча и количество эпох.
   * __BatchNormalization__ (nn.BatchNorm2d) — ключ к победе!
     * Иногда чем больше батч-нормализации, тем лучше.
   * __Регуляризация__ для предотвращения переобучения
     * Добавьте L2-норму весов в функцию потерь, PyTorch сделает остальное
       * Это можно сделать вручную или с помощью параметра `weight_decay` оптимизатора ([например, документация SGD](https://pytorch.org/docs/stable/optim.html#torch.optim.SGD)).
     * Dropout (`nn.Dropout`) — для предотвращения переобучения
       * Не переусердствуйте. Проверьте, действительно ли он улучшает вашу сеть.
   
### Сверточные архитектуры
   * Эту задачу __можно__ решить последовательностью сверток и пулингов с приправой из batch_norm и ReLU, но вам не обязательно на этом останавливаться.
   * [Семейство Inception](https://hacktilldawn.com/2016/09/25/inception-modules-explained-and-implemented/), [семейство ResNet](https://towardsdatascience.com/an-overview-of-resnet-and-its-variants-5281e2f56035?gi=9018057983ca), [плотно-связанные свертки (экзотика)](https://arxiv.org/abs/1608.06993), [капсульные сети (экзотика)](https://arxiv.org/abs/1710.09829)
   * Пожалуйста, попробуйте несколько простых архитектур, прежде чем браться за resnet-152.
   * Внимание! Обучение сверточных сетей может занять много времени без GPU. Это нормально. **Однако крайне рекомендуется ипользовать gpu в colab'e**
     * Делайте разумные оценки размера слоев. Первая свертка со 128 нейронами — это, скорее всего, перебор.
     * __Чтобы сократить время вычислений__ в несколько раз в обмен на некоторое падение точности, попробуйте использовать параметр __stride__. Свертка с `stride=2` должна занимать примерно 1/4 времени от свертки по умолчанию (`stride=1`).

   
### Аугментация данных
   * получить датасет в 5 раз больше бесплатно — это здорово
     * Увеличение+обрезка = сдвиг
     * Поворот+увеличение (чтобы убрать черные полосы)
     * Добавление шума (гауссовского или бернулли)
   * Простой способ сделать это (если у вас есть PIL/Image):
     * ```from scipy.misc import imrotate,imresize```
     * и немного нарезки (slicing)
     * Другие крутые библиотеки: cv2, scikit-image, PIL/Pillow
   * Более продвинутый способ — использовать трансформации torchvision:
    ```
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(root=путь_к_cifar_как_в_семинаре, train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

    ```
   * Или используйте этот инструмент из Keras (требует Theano/TensorFlow): [туториал](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html), [документация](https://keras.io/preprocessing/image/)
   * Будьте реалистами. Обычно нет смысла переворачивать собак вверх ногами, так как в жизни вы их так не видите.
   
```

```

```

```

```

```

```

```


In [ ]:
# вы можете написать свое решение прямо здесь :)